In [1]:
import requests
import json
import uuid
from typing import List, Dict
from datetime import datetime
import time

# --- KHAI BÁO BIẾN CẤU HÌNH ---
BASE_URL = "http://localhost:8000"

# (Giả định bạn có thể import 3 components chính này để test nội bộ)
# from vectordb import VectorDBClient 
# from clients import FPTAIClient
# from router import SemanticRouter 

print(f"Server Test Base URL: {BASE_URL}")
print("Vui lòng đảm bảo Server FastAPI đang chạy.")

Server Test Base URL: http://localhost:8000
Vui lòng đảm bảo Server FastAPI đang chạy.


In [2]:
print("--- 2.1. Chuẩn bị dữ liệu mẫu ---")

# Giả định cấu trúc data mà bạn cần gửi vào db_client.add_jobs()
DUMMY_JOBS = [
    {
        "id": str(uuid.uuid4()),
        "description": "Cần 1 lập trình viên biết Java, Spring Boot, microservices. Kinh nghiệm 5 năm, yêu cầu tiếng Anh tốt.",
        "category": "IT Software",
        "metadata": {"company": "FPT", "location": "Hà Nội"}
    },
    {
        "id": str(uuid.uuid4()),
        "description": "Tuyển dụng nhân viên Kế toán tổng hợp. Yêu cầu thành thạo MISA, báo cáo thuế. Ưu tiên kinh nghiệm 3 năm.",
        "category": "Kế toán",
        "metadata": {"company": "KPMG", "location": "TP.HCM"}
    },
    {
        "id": str(uuid.uuid4()),
        "description": "Marketing Manager có kinh nghiệm chạy quảng cáo Meta, Google Ads. Yêu cầu sáng tạo và phân tích data tốt.",
        "category": "Marketing",
        "metadata": {"company": "Shopee", "location": "Hà Nội"}
    }
]

# Lưu ý: Thay vì gọi trực tiếp add_jobs(), ta sẽ gọi qua API nếu có
# Hoặc nếu test nội bộ, bạn gọi hàm sau: 
# db_client.add_jobs(DUMMY_JOBS)
# Sau khi chạy đoạn này thành công, folder chroma_data/ sẽ được tạo.

print(f"✅ Đã tạo {len(DUMMY_JOBS)} Jobs mẫu. Kiểm tra folder chroma_data/.")

--- 2.1. Chuẩn bị dữ liệu mẫu ---
✅ Đã tạo 3 Jobs mẫu. Kiểm tra folder chroma_data/.


In [3]:
print("\n--- 3.1. Test Semantic Routing (Streaming) ---")

# Mẫu câu hỏi đã được định nghĩa trong respond.json (Intent: 'salary_neg')
KNOWN_QUERY = "Làm sao để deal lương cao khi phỏng vấn lần đầu?"
UNKNOWN_QUERY = "Hôm nay thời tiết thế nào ở Hà Nội?"

def test_streaming_chat(query: str, endpoint: str = "/chat/general"):
    print(f"\nQUERY: {query}")
    url = f"{BASE_URL}{endpoint}"
    payload = {"text": query}
    
    # Sử dụng logic stream để nhận từng token
    with requests.post(url, json=payload, stream=True) as response:
        response.raise_for_status() # Kiểm tra lỗi HTTP
        
        full_response = ""
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                # Dùng end="" và flush=True để mô phỏng hiệu ứng gõ chữ
                text = chunk.decode("utf-8")
                print(text, end="", flush=True)
                full_response += text
        
        if "chuyên gia đàm phán" in full_response:
             print("\n-> Kết quả: 🎯 Đã kích hoạt chế độ CHUYÊN GIA (Routing thành công)")
        else:
             print("\n-> Kết quả: 🌍 Đã kích hoạt chế độ CHUNG (Routing thành công)")

# Test 1: Câu hỏi biết trước (nên kích hoạt chuyên gia)
test_streaming_chat(KNOWN_QUERY)

# Test 2: Câu hỏi lạ (nên dùng chế độ chung)
test_streaming_chat(UNKNOWN_QUERY)


--- 3.1. Test Semantic Routing (Streaming) ---

QUERY: Làm sao để deal lương cao khi phỏng vấn lần đầu?


ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /chat/general (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002217A241090>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
print("\n--- 4.1. Test Pipeline RAG (Mô phỏng Ứng viên) ---")

SAMPLE_CV = """
Tôi tên là Nguyễn Văn A. Tốt nghiệp Đại học Bách Khoa Hà Nội, chuyên ngành Công nghệ thông tin. 
Có 6 năm kinh nghiệm làm việc với Java, chủ yếu dùng framework Spring Boot và quản lý dự án với Agile. 
Tôi có chứng chỉ AWS Solution Architect Associate và hiện đang muốn tìm công việc IT ở Hà Nội.
"""

def test_rag_pipeline(cv_text: str):
    print("\n\n--- BẮT ĐẦU QUY TRÌNH TƯ VẤN RAG ---")
    url = f"{BASE_URL}/pipeline/consult"
    payload = {"raw_cv": cv_text}
    
    # RAG là quá trình 3 bước, cần thời gian, nên cũng dùng Streaming
    with requests.post(url, json=payload, stream=True) as response:
        response.raise_for_status()
        
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                text = chunk.decode("utf-8")
                print(text, end="", flush=True)
                
# Chạy RAG Pipeline với CV mẫu (Nó sẽ tìm thấy Job Java ở Cell 2)
test_rag_pipeline(SAMPLE_CV)